# Flight Trend & Optimization Analyzer

## 1. Project Overview

This project fetches **real-time global flight data** from the [OpenSky Network API](https://opensky-network.org) and performs data analysis using Object-Oriented Programming (OOP) in Python.

**Core objectives:**
1. Fetch live flight data via REST API
2. Analyze country distribution and airline activity
3. Study velocity and altitude patterns
4. Filter and export data for further use


---
## 2. Environment Setup & Module Import


In [1]:
import pandas as pd
from src.flight_logic import FlightDataAnalyzer, LiveFlightAPI

print("[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.")


[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.


---
## 3. Load Live Data from OpenSky Network API

We use the `LiveFlightAPI` child class (inherits from `FlightDataAnalyzer`) to fetch real-time flight data.
The endpoint `https://opensky-network.org/api/states/all` delivers a dynamic **JSON response**:
- The root object contains `time` (Unix timestamp) and `states` (a 2D array of airborne aircraft state vectors).
- Because `states` contains an array of raw values rather than named keys, our ingestion logic parses the official index positions:
  - `flight[0]`: **icao24** (unique 24-bit transponder hex ID)
  - `flight[1]`: **callsign** (8-character flight identifier)
  - `flight[2]`: **country** (aircraft country of registration)
  - `flight[5]`: **longitude** & `flight[6]`: **latitude** (WGS-84 coordinates)
  - `flight[7]`: **altitude** (barometric altitude in meters)
  - `flight[9]`: **velocity** (ground speed in m/s)
  - `flight[10]`: **heading** (direction of motion in degrees)

The parsed records are converted into a `pandas.DataFrame` and cleaned of missing transponder records.


In [2]:
# Initialize API loader and fetch live flight data
live_api = LiveFlightAPI()

if live_api.fetch_live_flights(limit=500):
    print(f"[SUCCESS] Loaded {len(live_api.df)} live flights from OpenSky Network")
    print(f"Columns: {list(live_api.df.columns)}")
    display(live_api.df.head(10))
else:
    print("[ERROR] Failed to fetch data. Check network connection.")


[SUCCESS] Loaded 500 live flights from OpenSky Network
Columns: ['icao24', 'callsign', 'country', 'longitude', 'latitude', 'altitude', 'velocity', 'heading']


,icao24,callsign,country,longitude,latitude,altitude,velocity,heading
0,50047c,N/A,San Marino,NaN,NaN,NaN,239.88,111.78
1,39de4e,TVF93EP,France,3.0731,39.8675,2026.92,149.62,175.66
2,e8027d,LPE2465,Chile,-60.3068,-33.5882,10279.38,239.45,149.25
3,39de4b,TVF44UX,France,25.2143,39.4717,10972.80,209.21,325.22
4,39de4a,TVF49CW,France,-5.9216,53.4016,1021.08,96.95,273.96
5,39de4d,TVF51XP,France,3.1483,42.1630,10332.72,232.97,168.02
6,801642,AXB1071,India,85.0869,26.4717,10972.80,240.13,285.41
7,39de59,TVF38EG,France,7.4364,46.0225,10965.18,215.04,304.37
8,39de58,TVF12RK,France,2.3772,48.7271,NaN,5.40,45.00
9,3cf0b8,PTN26Z,Germany,6.1619,46.0811,4602.48,154.42,209.98


---
## 4. Feature 1: Country Distribution Analysis

Which countries have the most airborne flights right now?


In [3]:
# Top 10 countries by live flight count (tabular analysis)
country_stats = live_api.df["country"].value_counts().head(10)

print("=== Top 10 Countries by Live Flight Count ===")
for country, count in country_stats.items():
    print(f"  {country}: {count} flights")


=== Top 10 Countries by Live Flight Count ===
  India: 62 flights
  Sweden: 56 flights
  United States: 46 flights
  Turkey: 38 flights
  Hungary: 37 flights
  Germany: 36 flights
  Austria: 31 flights
  France: 25 flights
  Spain: 23 flights
  Greece: 13 flights


---
## 5. Feature 2: Velocity & Altitude Analysis

Analyze flight speed and altitude patterns across countries.


In [4]:
# Compute velocity and altitude stats by country
valid_flights = live_api.df.dropna(subset=["velocity", "altitude"])

vel_stats = valid_flights.groupby("country").agg(
    flight_count=("icao24", "count"),
    avg_velocity=("velocity", "mean"),
    avg_altitude=("altitude", "mean"),
    max_velocity=("velocity", "max")
).reset_index()

vel_stats["avg_velocity"] = vel_stats["avg_velocity"].round(2)
vel_stats["avg_altitude"] = vel_stats["avg_altitude"].round(2)
vel_stats["max_velocity"] = vel_stats["max_velocity"].round(2)

top_vel = vel_stats.sort_values("flight_count", ascending=False).head(10)
print("=== Top 10 Countries: Velocity & Altitude Stats ===")
display(top_vel)


=== Top 10 Countries: Velocity & Altitude Stats ===


,country,flight_count,avg_velocity,avg_altitude,max_velocity
15,India,58,190.96,6958.64,273.79
30,Sweden,52,205.34,8489.85,254.07
36,United States,41,195.42,7635.24,290.84
11,Germany,36,159.09,6309.36,257.61
13,Hungary,36,257.60,8827.56,1964.93
33,Turkey,34,210.03,9269.51,252.64
2,Austria,29,175.94,7818.91,236.29
10,France,24,151.50,6094.73,247.86
29,Spain,18,201.44,8213.94,242.96
4,Brazil,12,172.62,6326.50,237.22


In [ ]:
live_api.show_velocity_stats()
